# SMARD (Bundesnetzagentur)

Required data:
1. Actual Load (Gesamtverbrauch)
2. Residual Load (Residuallast)
3. Wind Onshore Actual & Forecast
4. Wind Offshore Actual & Forecast
5. Photovoltaic Actual & Forecast

Ressources: 

- [SMARD Swagger API](https://smard.api.bund.dev/)

Pull all SMARD data with the following terminal command: `python3 -m ingestion.fetch_smard --start 2022-01-01 --end 2025-12-31`

In [ ]:
import requests
import pandas as pd
import time
from datetime import datetime
import io

# --- KONFIGURATION ---
# WICHTIG: Physikalische Daten (Last, Erzeugung) liegen unter "DE", nicht "DE-LU"
REGION = "DE-LU"  
RESOLUTION = "hour"
START_DATE = pd.Timestamp("2022-01-01").tz_localize("UTC")
END_DATE = pd.Timestamp("2025-12-31").tz_localize("UTC")

# Die korrekte Base-URL für den Datenabruf
BASE_URL = "https://www.smard.de/app/chart_data"

# Mapping der Module
DATA_MODULES = {
    # ACTUALS (Ist-Werte)
    "load_actual": 410,             # Gesamtverbrauch (Netzlast)
    "residual_load_actual": 4359,   # Residuallast
    "wind_onshore_actual": 4067,    # Wind Onshore Erzeugung
    "wind_offshore_actual": 1225,   # Wind Offshore Erzeugung
    "solar_actual": 4068,           # PV Erzeugung
    
    # FORECASTS (Prognosen)
    "wind_onshore_forecast": 123,   # Prognose Onshore
    "wind_offshore_forecast": 3791, # Prognose Offshore
    "solar_forecast": 125           # Prognose PV
}

def get_available_timestamps(filter_id, region, resolution):
    """Holt die Liste der verfügbaren Zeitstempel."""
    url = f"{BASE_URL}/{filter_id}/{region}/index_{resolution}.json"
    try:
        response = requests.get(url)
        if response.status_code == 404:
            print(f"WARNUNG: URL nicht gefunden: {url}")
            return []
        response.raise_for_status()
        return sorted(response.json()["timestamps"])
    except Exception as e:
        print(f"Fehler beim Index-Abruf ID {filter_id}: {e}")
        return []

def fetch_data_chunk(filter_id, region, resolution, timestamp):
    """Lädt einen Daten-Chunk."""
    # Dateiname Format: {filter}_{region}_{resolution}_{timestamp}.json
    filename = f"{filter_id}_{region}_{resolution}_{timestamp}.json"
    url = f"{BASE_URL}/{filter_id}/{region}/{filename}"
    try:
        response = requests.get(url)
        response.raise_for_status()
        return response.json()["series"]
    except Exception as e:
        print(f"Fehler bei Chunk {timestamp}: {e}")
        return []

def download_smard_data():
    all_dataframes = []
    print(f"Starte Download von {BASE_URL} für Region {REGION}...\n")

    # Wir suchen Zeitstempel ab Ende 2021, um den Start 2022 sicher zu erwischen
    cutoff_ms = (START_DATE - pd.DateOffset(months=2)).timestamp() * 1000

    for col_name, filter_id in DATA_MODULES.items():
        print(f"Lade: {col_name} (ID: {filter_id})...")
        
        timestamps = get_available_timestamps(filter_id, REGION, RESOLUTION)
        relevant_timestamps = [ts for ts in timestamps if ts >= cutoff_ms]

        if not relevant_timestamps:
            print(f" -> KEINE DATEN gefunden für ID {filter_id} (Region: {REGION})")
            continue

        series_data = []
        for ts in relevant_timestamps:
            chunk = fetch_data_chunk(filter_id, REGION, RESOLUTION, ts)
            series_data.extend(chunk)
            
        if not series_data:
            continue

        # DataFrame erstellen
        df = pd.DataFrame(series_data, columns=["timestamp_ms", col_name])
        df["timestamp"] = pd.to_datetime(df["timestamp_ms"], unit="ms", utc=True)
        df.drop(columns=["timestamp_ms"], inplace=True)
        
        # Auf Zeitraum filtern
        df = df[(df["timestamp"] >= START_DATE) & (df["timestamp"] <= END_DATE)]
        
        # Duplikate entfernen & Index
        df.drop_duplicates(subset="timestamp", inplace=True)
        df.set_index("timestamp", inplace=True)
        
        all_dataframes.append(df)
        print(f" -> {len(df)} Zeilen geladen.")

    if all_dataframes:
        print("\nFüge Daten zusammen...")
        final_df = pd.concat(all_dataframes, axis=1)
        final_df.sort_index(inplace=True)
        
        # Speichern
        csv_name = "smard_data_actuals_2022_2025.csv"
        final_df.to_csv(csv_name)
        print(f"ERFOLG! Datei gespeichert: {csv_name}")
        print(final_df.head())
        return final_df
    else:
        print("Es konnten keine Daten geladen werden.")
        return None

if __name__ == "__main__":
    download_smard_data()

Starte Download von https://www.smard.de/app/chart_data für Region DE...

Lade: load_actual (ID: 410)...
 -> 35041 Zeilen geladen.
Lade: residual_load_actual (ID: 4359)...
 -> 35041 Zeilen geladen.
Lade: wind_onshore_actual (ID: 4067)...
 -> 35041 Zeilen geladen.
Lade: wind_offshore_actual (ID: 1225)...
Fehler bei Chunk 1754258400000: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))
 -> 34873 Zeilen geladen.
Lade: solar_actual (ID: 4068)...
Fehler bei Chunk 1688335200000: ('Connection aborted.', ConnectionResetError(54, 'Connection reset by peer'))
 -> 34873 Zeilen geladen.
Lade: wind_onshore_forecast (ID: 123)...
 -> 35041 Zeilen geladen.
Lade: wind_offshore_forecast (ID: 3791)...
 -> 35041 Zeilen geladen.
Lade: solar_forecast (ID: 125)...
 -> 35041 Zeilen geladen.

Füge Daten zusammen...
ERFOLG! Datei gespeichert: smard_data_actuals_2022_2025.csv
                           load_actual  residual_load_actual  \
timestamp                      

In [ ]:
import requests
import pandas as pd

def get_smard_data(filter_id, region="DE-LU", resolution="hour"):
    # 1. Index abrufen, um den neuesten Zeitstempel zu erhalten
    index_url = f"https://www.smard.de/app/chart_data/{filter_id}/{region}/index_{resolution}.json"
    timestamps = requests.get(index_url).json()['timestamps']
    
    # 2. Daten für alle verfügbaren Chunks laden (hier beispielhaft der letzte Chunk)
    all_series = []
    for ts in timestamps[-5:]: # Die letzten 5 Chunks für historische Abdeckung
        data_url = f"https://www.smard.de/app/chart_data/{filter_id}/{region}/{filter_id}_{region}_{resolution}_{ts}.json"
        res = requests.get(data_url).json()
        all_series.extend(res['series'])
    
    df = pd.DataFrame(all_series, columns=['timestamp', 'value'])
    df['timestamp'] = pd.to_datetime(df['timestamp'], unit='ms', utc=True)
    return df.set_index('timestamp').sort_index()

# Beispiel-Abruf für DA-Preis und Wind-Prognose
da_price = get_smard_data(4169) # Marktpreis DE/LU
wind_onshore_fc = get_smard_data(123) # Prognose Onshore
wind_offshore_fc = get_smard_data(3791) # Prognose Offshore

# Zusammenführen und Wind aggregieren
df_final = da_price.rename(columns={'value': 'da_price_eur'})
df_final['wind_forecast_de'] = wind_onshore_fc['value'] + wind_offshore_fc['value']

print(df_final.head())